In [ ]:
import numpy as np
from astropy.wcs import WCS
from reproject import reproject_interp

NY, NX   = 256, 256
PIXSCALE = 1.0 / 3600

np.random.seed(42)
image = np.random.rand(NY, NX)

w = WCS(naxis=2)
w.wcs.crpix = [NX / 2, NY / 2]
w.wcs.cdelt = [-PIXSCALE, PIXSCALE]
w.wcs.crval = [0.0, 0.0]
w.wcs.ctype = ['RA---TAN', 'DEC--TAN']
native_hdr = w.to_header()

native_sum = np.sum(image)
print(f"Native  {NY}x{NX}  sum = {native_sum:.4f}")
print()

for BF in [1, 2, 4, 8, 16]:
    blocked_shape = (NY // BF, NX // BF)
    blocked_hdr   = native_hdr.copy()
    blocked_hdr['CDELT1'] = native_hdr['CDELT1'] * BF
    blocked_hdr['CDELT2'] = native_hdr['CDELT2'] * BF
    blocked_hdr['CRPIX1'] = native_hdr['CRPIX1'] / BF
    blocked_hdr['CRPIX2'] = native_hdr['CRPIX2'] / BF

    blocked, _ = reproject_interp((image, native_hdr), blocked_hdr, shape_out=blocked_shape)
    blocked = np.nan_to_num(blocked)

    blocked_sum = np.sum(blocked)
    print(f"BF={BF:>2}   {blocked_shape[0]}x{blocked_shape[1]}   sum = {blocked_sum:.4f}   "
          f"native/blocked = {native_sum/blocked_sum:.4f}   "
          f"BF={BF}  BF^2={BF**2}")
